# Risk Scoring System — Phase 1 (Bank Transaction Dataset): Honest Data Audit

**Dataset:** Bank Transaction Dataset for Fraud Detection (valakhorasani, Kaggle), 2,512 rows.

**Why this dataset:** Our task is **risk scoring**, not fraud classification. A risk-scoring engine does not need a fraud label — it scores deviation from each account's normal behaviour. This dataset is the only public one that exposes all 8 of the mentor's requested features as usable columns (including AccountBalance and per-account Location), which Sparkov could not.

**What this notebook does NOT do:** build any scoring logic. We only audit whether each of the 8 features actually carries usable variation. We already saw warning signs earlier (PreviousTransactionDate looked artifactual; LoginAttempts looked constant). We verify everything honestly before building.

**The 8 features the mentor wants, mapped to columns:**
| # | Mentor's feature | Column(s) |
|---|---|---|
| 1 | Average transaction (30d) | TransactionAmount + TransactionDate |
| 2 | Maximum transaction ever | TransactionAmount |
| 3 | Transactions today | TransactionDate / PreviousTransactionDate |
| 4 | Typical transaction hours | TransactionDate (hour) |
| 5 | Common locations | Location |
| 6 | Common merchants | MerchantID |
| 7 | Account age | CustomerAge (or account history span) |
| 8 | Average balance | AccountBalance |

This notebook checks each one for real, usable variation.

## 0. Setup

In [1]:
import pandas as pd
import numpy as np

# Update path/filename to match your file
PATH = r"D:\Nithilan\Internship\QPay Internship\risk_scoring\data\bank_transactions.csv"

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

## 1. Load & Basic Shape

In [2]:
df = pd.read_csv(PATH)

print(f"Rows    : {len(df):,}")
print(f"Columns : {df.shape[1]}")
print(f"\nColumns:")
for c in df.columns:
    print(f"  - {c}")
print("\nFirst 3 rows:")
print(df.head(3).T)

Rows    : 2,512
Columns : 16

Columns:
  - TransactionID
  - AccountID
  - TransactionAmount
  - TransactionDate
  - TransactionType
  - Location
  - DeviceID
  - IP Address
  - MerchantID
  - Channel
  - CustomerAge
  - CustomerOccupation
  - TransactionDuration
  - LoginAttempts
  - AccountBalance
  - PreviousTransactionDate

First 3 rows:
                                           0                    1                    2
TransactionID                       TX000001             TX000002             TX000003
AccountID                            AC00128              AC00455              AC00019
TransactionAmount                      14.09               376.24               126.29
TransactionDate          2023-04-11 16:29:14  2023-06-27 16:44:19  2023-07-10 18:16:08
TransactionType                        Debit                Debit                Debit
Location                           San Diego              Houston                 Mesa
DeviceID                             D000380   

## 2. Nulls, Duplicates, Dtypes

In [3]:
print("Dtypes:")
print(df.dtypes)
print(f"\nNulls per column:")
n = df.isnull().sum()
print(n[n>0] if n.sum()>0 else "  None")
print(f"\nDuplicate rows: {df.duplicated().sum()}")

Dtypes:
TransactionID               object
AccountID                   object
TransactionAmount          float64
TransactionDate             object
TransactionType             object
Location                    object
DeviceID                    object
IP Address                  object
MerchantID                  object
Channel                     object
CustomerAge                  int64
CustomerOccupation          object
TransactionDuration          int64
LoginAttempts                int64
AccountBalance             float64
PreviousTransactionDate     object
dtype: object

Nulls per column:
  None

Duplicate rows: 0


## 3. Per-Account History — can we even build baselines?\nA behavioural baseline needs several transactions per account. If most accounts have only 1-2 transactions, per-account profiling is impossible. This is the first make-or-break check.

In [4]:
per_acct = df.groupby('AccountID').size()
print("Transactions per account:")
print(f"  Unique accounts : {df['AccountID'].nunique():,}")
print(f"  Min   : {per_acct.min()}")
print(f"  25th  : {per_acct.quantile(0.25):.0f}")
print(f"  Median: {per_acct.median():.0f}")
print(f"  Mean  : {per_acct.mean():.2f}")
print(f"  Max   : {per_acct.max()}")
print(f"\nAccounts with only 1 transaction: {(per_acct==1).sum()}")
print(f"Accounts with >=5 transactions   : {(per_acct>=5).sum()}")

Transactions per account:
  Unique accounts : 495
  Min   : 1
  25th  : 3
  Median: 5
  Mean  : 5.07
  Max   : 12

Accounts with only 1 transaction: 24
Accounts with >=5 transactions   : 285


## 4. FEATURE 8 — AccountBalance: does it vary sensibly?\nThis is the feature Sparkov lacked entirely and the main reason we switched. Verify it's real and varies.

In [5]:
print("AccountBalance summary:")
print(df['AccountBalance'].describe().round(2))
print(f"\nUnique balance values: {df['AccountBalance'].nunique()}")
print(f"Constant? {df['AccountBalance'].nunique()==1}")
# does balance vary WITHIN an account (needed for 'average balance maintained')?
bal_var = df.groupby('AccountID')['AccountBalance'].nunique()
print(f"\nAccounts where balance changes across txns: {(bal_var>1).sum()} / {df['AccountID'].nunique()}")

AccountBalance summary:
count     2512.00
mean      5114.30
std       3900.94
min        101.25
25%       1504.37
50%       4735.51
75%       7678.82
max      14977.99
Name: AccountBalance, dtype: float64

Unique balance values: 2510
Constant? False

Accounts where balance changes across txns: 471 / 495


## 5. FEATURE 5 — Location: do accounts have MULTIPLE locations?\nThis is THE feature that killed Sparkov (every card = 1 city). The whole reason for switching. If accounts here also have a single location each, this dataset has the same fatal flaw and we must know now.

In [6]:
loc_per_acct = df.groupby('AccountID')['Location'].nunique()
print(f"Distinct locations per account:")
print(f"  Min   : {loc_per_acct.min()}")
print(f"  Median: {loc_per_acct.median():.0f}")
print(f"  Max   : {loc_per_acct.max()}")
print(f"\nAccounts with only 1 location : {(loc_per_acct==1).sum()} / {len(loc_per_acct)}")
print(f"Accounts with multiple locations: {(loc_per_acct>1).sum()} / {len(loc_per_acct)}")
print(f"\nTotal distinct locations in data: {df['Location'].nunique()}")
print("\nVERDICT:")
if (loc_per_acct>1).mean() > 0.5:
    print("  Most accounts transact in MULTIPLE locations -> location feature VIABLE.")
else:
    print("  Most accounts have a SINGLE location -> same flaw as Sparkov. Location feature weak/dead.")

Distinct locations per account:
  Min   : 1
  Median: 5
  Max   : 11

Accounts with only 1 location : 25 / 495
Accounts with multiple locations: 470 / 495

Total distinct locations in data: 43

VERDICT:
  Most accounts transact in MULTIPLE locations -> location feature VIABLE.


## 6. FEATURE 6 — Merchant: preferred vs outlier merchants per account?\nThe dataset description claims it models 'preferred and outlier merchants per account'. Verify accounts have a mix — a dominant set plus occasional new ones.

In [7]:
merch_per_acct = df.groupby('AccountID')['MerchantID'].nunique()
print(f"Distinct merchants per account:")
print(f"  Min   : {merch_per_acct.min()}")
print(f"  Median: {merch_per_acct.median():.0f}")
print(f"  Max   : {merch_per_acct.max()}")
print(f"\nTotal distinct merchants: {df['MerchantID'].nunique()}")
print(f"Accounts with >1 merchant: {(merch_per_acct>1).sum()} / {len(merch_per_acct)}")

Distinct merchants per account:
  Min   : 1
  Median: 5
  Max   : 12

Total distinct merchants: 100
Accounts with >1 merchant: 469 / 495


## 7. FEATURE 4 — Typical hours: do transactions spread across hours, or all the same?\nSparkov failed here too (every card = all 24 hours). Check whether hour carries per-account signal.

In [8]:
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'], errors='coerce')
df['hour'] = df['TransactionDate'].dt.hour
print(f"Parse failures in TransactionDate: {df['TransactionDate'].isna().sum()}")
print("\nOverall hour distribution:")
print(df['hour'].value_counts().sort_index())
hours_per_acct = df.groupby('AccountID')['hour'].nunique()
print(f"\nDistinct hours per account: median {hours_per_acct.median():.0f}, max {hours_per_acct.max()}")

Parse failures in TransactionDate: 0

Overall hour distribution:
hour
16    1316
17     819
18     377
Name: count, dtype: int64

Distinct hours per account: median 2, max 3


## 8. FEATURE 3 — PreviousTransactionDate: usable or artifact?\nEarlier we saw all values looked like 04-11-2024 seconds apart. Verify honestly whether this column is usable for transaction frequency.

In [9]:
if 'PreviousTransactionDate' in df.columns:
    df['PreviousTransactionDate'] = pd.to_datetime(df['PreviousTransactionDate'], errors='coerce')
    print("PreviousTransactionDate range:")
    print(f"  Min: {df['PreviousTransactionDate'].min()}")
    print(f"  Max: {df['PreviousTransactionDate'].max()}")
    print(f"  Unique dates (day level): {df['PreviousTransactionDate'].dt.date.nunique()}")
    # gap between current and previous
    gap = (df['TransactionDate'] - df['PreviousTransactionDate']).dt.total_seconds()
    print(f"\nGap (current - previous) in seconds:")
    print(pd.Series(gap).describe())
    print("\nIf gaps are all negative or near-constant, this column is an artifact.")
else:
    print("No PreviousTransactionDate column.")

PreviousTransactionDate range:
  Min: 2024-11-04 08:06:23
  Max: 2024-11-04 08:12:23
  Unique dates (day level): 1

Gap (current - previous) in seconds:
count    2.512000e+03
mean    -4.211863e+07
std      9.191205e+06
min     -5.803267e+07
25%     -5.016886e+07
50%     -4.195566e+07
75%     -3.408996e+07
max     -2.657429e+07
dtype: float64

If gaps are all negative or near-constant, this column is an artifact.


## 9. FEATURE 7 — CustomerAge & FEATURE 1/2 — Amount

In [10]:
print("CustomerAge:")
print(df['CustomerAge'].describe().round(1))
print(f"\nTransactionAmount:")
print(df['TransactionAmount'].describe().round(2))
amt_per_acct = df.groupby('AccountID')['TransactionAmount'].agg(['mean','std','max','count'])
print(f"\nPer-account amount stats (sample):")
print(amt_per_acct.head())

CustomerAge:
count    2512.0
mean       44.7
std        17.8
min        18.0
25%        27.0
50%        45.0
75%        59.0
max        80.0
Name: CustomerAge, dtype: float64

TransactionAmount:
count    2512.00
mean      297.59
std       291.95
min         0.26
25%        81.89
50%       211.14
75%       414.53
max      1919.11
Name: TransactionAmount, dtype: float64

Per-account amount stats (sample):
                 mean         std     max  count
AccountID                                       
AC00001    130.380000  116.799898  212.97      2
AC00002    293.744286  195.695091  516.47      7
AC00003    253.268000  158.484753  416.62      5
AC00004    242.231111  231.081586  642.54      9
AC00005    347.974444  189.487268  680.76      9


## 10. Other columns — LoginAttempts, Channel, TransactionType\nCheck if LoginAttempts is constant (we suspected =1 always) and whether Channel/Type add signal.

In [11]:
for col in ['LoginAttempts','Channel','TransactionType']:
    if col in df.columns:
        print(f"{col}: {df[col].nunique()} unique -> {df[col].value_counts().to_dict()}")
        print()

LoginAttempts: 5 unique -> {1: 2390, 5: 32, 4: 32, 3: 31, 2: 27}

Channel: 3 unique -> {'Branch': 868, 'ATM': 833, 'Online': 811}

TransactionType: 2 unique -> {'Debit': 1944, 'Credit': 568}



## 11. AUDIT VERDICT — fill in from REAL outputs above

| # | Feature | Column | Verdict (viable / weak / dead) | Evidence |
|---|---|---|---|---|
| 1 | Avg txn 30d | TransactionAmount+Date | ? | ? |
| 2 | Max txn ever | TransactionAmount | ? | ? |
| 3 | Txns today | PreviousTransactionDate | ? | ? |
| 4 | Typical hours | hour | ? | ? |
| 5 | Common locations | Location | ? | per-account location count |
| 6 | Common merchants | MerchantID | ? | per-account merchant count |
| 7 | Account age | CustomerAge | ? | ? |
| 8 | Avg balance | AccountBalance | ? | varies within account? |

**The two make-or-break cells:**
- **Cell 3** — enough transactions per account to build a baseline at all?
- **Cell 5** — do accounts have multiple locations? (this is what Sparkov failed)

If Cell 3 shows most accounts have very few transactions, OR Cell 5 shows single-location accounts again, this dataset has the same problems and we reassess honestly rather than forcing it.